---
title: "Switzerland's Export Destinations"
subtitle: "Annual partner-total discovery, 2023–2025"
lang: en
format:
  pdf:
    documentclass: scrartcl
    papersize: a4
    fontsize: 11pt
    geometry:
      - margin=25mm
    toc: true
    toc-depth: 2
    number-sections: true
    colorlinks: true
    code-overflow: wrap
execute:
  enabled: false
  echo: true
---

# Summary and V2 scope

This V2 notebook is a separate branch of the Swiss Comtrade project. It begins by collecting Switzerland's annual export totals by partner area for 2023–2025. The original `UN_commodity.ipynb` remains the World-by-chapter baseline.

The partner discovery query covers Switzerland (Comtrade reporter code **757**), annual goods exports, H6 classification, and all partner areas. It returns each reported partner area and the World aggregate. The next MongoDB stage will rank partner areas; after the user authorizes that stage, a later request will retrieve HS4 detail for the selected top 30.

Planned analysis after those stages:

1. Which partner areas have the largest total export values across 2023–2025?
2. Which HS4 headings have the largest export values within those top 30 partners?
3. How do partner and product shares change across the three years?
4. How concentrated are exports across partner areas and HS4 headings?

`primaryValue` is in current US dollars. It measures trade value, not physical volume. Partner areas may include territories or special aggregates, not only sovereign countries.

**Current V2 workflow:** request partner totals → check and save the discovery data.

**Next stage:** the cells below support HS4 extraction, snapshot saving, raw loading, and top-50 ranking. New API and database operations require explicit opt-in. The nested clean collection and research-question analysis remain future work; unexecuted cells do not establish results.

# Requirements and configuration

Use a Python kernel with `requests`, `pandas`, `python-dotenv`, `pymongo`, and `dnspython` installed. Install the repository-local dependencies with `python -m pip install -r requirements.txt` from this notebook's folder. Use the same Python environment as the notebook kernel. `python-dotenv` loads local configuration; `pymongo` connects to MongoDB, and `dnspython` supports Atlas SRV address lookup.

The course template uses `requests.get()`, `.json()`, a list of records, and simple loops with a delay. V2 keeps those patterns. Discovery uses `partnerCode=None`, which omits the partner filter, and `cmdCode="TOTAL"`. HS4 extraction later uses individual partner codes and `cmdCode="AG4"`.

In [7]:
import os
import time
import json
import math
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from pprint import pprint

import pandas as pd
import pymongo
import requests
from dotenv import load_dotenv

# Keep notebook tables compact enough for reading and later PDF export.
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", 60)

## Reproduction options

**Option A — fresh partner discovery:** leave `DATA_SOURCE = "api"`. V2 requests the annual partner totals and saves a dated snapshot after validation.

**Option B — saved partner discovery snapshot:** set `DATA_SOURCE = "snapshot"` and set `SNAPSHOT_PATH` to the exact V2 partner-summary JSON file. No API key or network request is needed in this mode; the query scope and record structure are checked before using its records.

V2 snapshots use a different query scope from the V1 World-by-chapter snapshots. Do not reuse a V1 snapshot as V2 partner data.

In [2]:
DATA_SOURCE = "snapshot" # snapshot or api
SNAPSHOT_PATH = Path("data/v2_partner_totals_20261016T143600945247Z.json")

if DATA_SOURCE not in {"api", "snapshot"}:
    raise ValueError("Choose either api or snapshot as DATA_SOURCE.")

## API key

**Only needed for fresh extraction; skipped in snapshot mode.** Use the registered UN Comtrade key already configured for this project. Keep the key in the local `.env` file or use the hidden prompt; do not copy it into the notebook, outputs, snapshots, or CSV files.

V2 uses the authenticated `/data/v1/get` endpoint. See the official [UN Comtrade API documentation](https://comtradeplus.un.org/ApiDocs) for current access and parameter behavior.

In [3]:
headers = {}
# Read the secret without writing it into the notebook source.
if DATA_SOURCE == "api":
    load_dotenv(dotenv_path=".env", override=False)
    api_key = os.environ.get("UN_COMTRADE_API_KEY", "").strip()
    if not api_key:
        api_key = getpass("UN Comtrade API key: ").strip()

    if not api_key:
        raise ValueError("An API key is required for this authenticated endpoint.")

    # A header keeps the key out of the query URL.
    headers = {"Ocp-Apim-Subscription-Key": api_key}
    print("API key loaded. Its value is not displayed.")
else:
    print("Snapshot mode: no API key is loaded or requested.")

Snapshot mode: no API key is loaded or requested.


## V2 partner-total query settings

| Setting | Meaning |
|---|---|
| `C / A / HS` in the endpoint | Goods, annual frequency, original HS classification |
| `reporterCode = 757` | Switzerland; Comtrade code, not ISO numeric code 756 |
| `partnerCode = None` | Omit the partner filter; return reported partner areas plus World |
| `cmdCode = "TOTAL"` | Total across commodity codes; this discovery stage has no HS4 breakdown |
| `flowCode = "X"` | Exports |
| `partner2Code = 0`, `customsCode = "C00"`, `motCode = 0` | Keep these additional dimensions at the project's aggregate selections |
| `maxRecords = 100000` | Free registered API per-call limit; V2 checks the response count and stops if a query reaches the cap |

The partner summary is a discovery dataset, not the final partner-by-product dataset. The `ALL` result includes a World row; keep it for context, but exclude it from partner rankings. Some partner-area codes may represent geographic or special aggregates.

### API parameter reference

| Parameter | Meaning | V2 value |
|---|---|---|
| `typeCode` | Goods (`C`) or services (`S`) | `C` in the URL |
| `freqCode` | Annual (`A`) or monthly (`M`) | `A` in the URL |
| `clCode` | Product classification | `HS` in the URL; returned edition must be `H6` |
| `period` | Calendar year | One year per request, 2023–2025 |
| `reporterCode` | Country/area reporting trade | `757`: Switzerland |
| `partnerCode` | Partner country/area | `None`: omit this filter and include World |
| `cmdCode` | Commodity selection | `TOTAL`: aggregate all commodity codes for destination discovery |
| `flowCode` | Trade direction | `X`: exports |
| `partner2Code` | Additional aggregate dimension | `0` |
| `customsCode`, `motCode` | Customs and transport selections | `C00`, `0` |

These are discovery totals. HS4 codes are requested only after the top partner areas have been selected in the later MongoDB phase.

In [8]:
API_URL = "https://comtradeapi.un.org/data/v1/get/C/A/HS"
YEARS = ["2023", "2024", "2025"]
MAX_RECORDS = 100000
REQUEST_DELAY = 5
EXPECTED_EDITION = "H6"

# These filters describe one annual all-commodity total per partner area.
base_params = {
    "reporterCode": 757,
    "partnerCode": None,
    "partner2Code": 0,
    "cmdCode": "TOTAL",
    "flowCode": "X",
    "customsCode": "C00",
    "motCode": 0,
    "maxRecords": MAX_RECORDS,
    "includeDesc": "true"
}

## Readable V2 query settings

Export one row of query parameters per year for provenance. These are request settings, not trade observations; they contain no API key.

In [5]:
Path("data").mkdir(exist_ok=True)
query_csv = Path("data") / f"v2_partner_query_settings_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.csv"
pd.DataFrame([{**base_params, "period": year} for year in YEARS]).to_csv(query_csv, index=False, mode="x")
print(f"V2 query settings saved to {query_csv}")

V2 query settings saved to data/v2_partner_query_settings_20261017T092929813727Z.csv


# ELT process

ELT means **Extract, Load, Transform**: collect source data, load it in raw form, then transform it in the target database. V2 first extracts partner-total discovery records. After MongoDB connection is authorized, aggregation pipelines will rank partners; a later API request will retrieve HS4 detail for the selected partners.

## Extract: request one year of partner totals

`requests.get()` sends the query. `.raise_for_status()` stops on an HTTP error, and `.json()` parses the response into Python dictionaries and lists. The API response includes a `count`; V2 checks it against the number of records in `data`. The `ALL` partner selection includes the World aggregate.

In [6]:
def request_comtrade_records(year, query_params, request_headers):
    params = query_params.copy()
    params["period"] = year
    for attempt in range(3):
        try:
            response = requests.get(
                API_URL, headers=request_headers, params=params, timeout=45
            )
        except requests.exceptions.Timeout:
            if attempt == 2:
                raise
            wait_seconds = 15 * (attempt + 1)
            print(f"Timeout for {year}; retrying in {wait_seconds} seconds.")
            time.sleep(wait_seconds)
            continue

        if response.status_code == 429 and attempt < 2:
            wait_seconds = 15 * (attempt + 1)
            print(f"Rate limit for {year}; retrying in {wait_seconds} seconds.")
            time.sleep(wait_seconds)
            continue

        if response.status_code == 400:
            try:
                error_payload = response.json()
                if isinstance(error_payload, dict):
                    error_detail = error_payload.get("error") or error_payload.get("message") or error_payload
                else:
                    error_detail = error_payload
            except ValueError:
                error_detail = response.text[:500]
            raise ValueError(f"Comtrade rejected the {year} query: {error_detail}")

        response.raise_for_status()
        payload = response.json()

        if not isinstance(payload, dict):
            raise ValueError("Expected a JSON object from Comtrade.")
        if payload.get("error"):
            raise ValueError("Comtrade reported an API error; review the query and documentation.")

        if not isinstance(payload.get("data"), list):
            raise ValueError("Expected a list of records in the API's data field.")

        records = payload["data"]
        count = payload.get("count")
        if isinstance(count, bool) or not isinstance(count, (int, str)):
            raise ValueError("Expected an integer record count from Comtrade.")
        try:
            api_count = int(count)
        except (TypeError, ValueError):
            raise ValueError("Expected an integer record count from Comtrade.") from None
        if api_count != len(records):
            raise ValueError("Comtrade's response count differs from its data list.")
        if api_count >= MAX_RECORDS:
            raise ValueError("The record cap was reached; partition the query before continuing.")

        return records, api_count


def extract_partner_totals(year):
    return request_comtrade_records(year, base_params, headers)

## Extract: collect the partner summaries

The loop makes one data request per year. `list_trade` holds the raw partner-area records; `extraction_log` records each year's API count, received count, query, and timestamp. This cell does not write to MongoDB. The later MongoDB stage will rank the partners.

In [7]:
list_trade = []
extraction_log = []
query_scope = {
    "endpoint": API_URL,
    "years": YEARS,
    "filters": base_params,
    "expected_edition": EXPECTED_EDITION,
    "partner_selection": "ALL",
    "commodity_selection": "TOTAL"
}

if DATA_SOURCE == "snapshot":
    with SNAPSHOT_PATH.open(encoding="utf-8") as file:
        snapshot = json.load(file)
    if not isinstance(snapshot, dict) or snapshot.get("schema_version") != 2:
        raise ValueError("Unsupported V2 partner-summary snapshot format.")
    if snapshot.get("query_scope") != query_scope:
        raise ValueError("Snapshot query settings differ from this V2 notebook's settings.")
    if not isinstance(snapshot.get("records"), list):
        raise ValueError("Snapshot must contain a list of raw partner records.")
    if not isinstance(snapshot.get("extraction_log"), list):
        raise ValueError("Snapshot must contain an extraction log.")
    list_trade = snapshot["records"]
    extraction_log = snapshot["extraction_log"]
    print(f"Loaded {len(list_trade)} partner-total records from {SNAPSHOT_PATH}.")
else:
    for i, year in enumerate(YEARS):
        print(f"Checking partner totals for {year} ({i + 1}/{len(YEARS)})")
        extracted_at = datetime.now(timezone.utc).isoformat()
        records, api_count = extract_partner_totals(year)

        list_trade.extend(records)
        extraction_log.append({
            "year_requested": year,
            "api_count": api_count,
            "records_received": len(records),
            "extracted_at_utc": extracted_at,
            "endpoint": API_URL,
            "query": {**base_params, "period": year}
        })

        if i < len(YEARS) - 1:
            time.sleep(REQUEST_DELAY)

    print(f"Partner discovery finished: {len(list_trade)} raw records.")

if not list_trade:
    raise ValueError("No partner-total records are available to inspect.")

Loaded 673 partner-total records from data/v2_partner_totals_20261016T143600945247Z.json.


## Inspect and validate the partner discovery records

Each record is one year's reported Swiss export total for a partner area. `partnerCode = 0` is the World aggregate, not an individual partner; retain it for context but exclude it from partner rankings. Partner reference codes can also represent areas or aggregates. This discovery extract has no HS4 breakdown yet.

In [8]:
# Inspect a raw trade document, not the secret-bearing request headers.
pprint(list_trade[0])

{'aggrLevel': 0,
 'altQty': 0.0,
 'altQtyUnitAbbr': 'N/A',
 'altQtyUnitCode': -1,
 'cifvalue': None,
 'classificationCode': 'H6',
 'classificationSearchCode': 'HS',
 'cmdCode': 'TOTAL',
 'cmdDesc': 'All Commodities',
 'customsCode': 'C00',
 'customsDesc': 'TOTAL CPC',
 'flowCode': 'X',
 'flowDesc': 'Export',
 'fobvalue': 419922543423.509,
 'freqCode': 'A',
 'grossWgt': 0.0,
 'isAggregate': True,
 'isAltQtyEstimated': False,
 'isGrossWgtEstimated': False,
 'isLeaf': False,
 'isNetWgtEstimated': False,
 'isOriginalClassification': True,
 'isQtyEstimated': False,
 'isReported': False,
 'legacyEstimationFlag': 0,
 'mosCode': '0',
 'motCode': 0,
 'motDesc': 'TOTAL MOT',
 'netWgt': None,
 'partner2Code': 0,
 'partner2Desc': 'World',
 'partner2ISO': 'W00',
 'partnerCode': 0,
 'partnerDesc': 'World',
 'partnerISO': 'W00',
 'period': '2023',
 'primaryValue': 419922543423.509,
 'qty': 0.0,
 'qtyUnitAbbr': 'N/A',
 'qtyUnitCode': -1,
 'refMonth': 52,
 'refPeriodId': 20230101,
 'refYear': 2023,
 'r

In [9]:
expected_fields = {
    "reporterCode": base_params["reporterCode"],
    "typeCode": "C",
    "freqCode": "A",
    "cmdCode": base_params["cmdCode"],
    "flowCode": base_params["flowCode"],
    "partner2Code": base_params["partner2Code"],
    "customsCode": base_params["customsCode"],
    "motCode": base_params["motCode"]
}

if sorted(entry.get("year_requested") for entry in extraction_log) != sorted(YEARS):
    raise ValueError("Extraction log does not cover each requested year exactly once.")

if sum(entry.get("records_received", 0) for entry in extraction_log) != len(list_trade):
    raise ValueError("Extraction-log record counts do not match the collected records.")
if any(
    entry.get("api_count") != entry.get("records_received")
    for entry in extraction_log
):
    raise ValueError("An API response count differs from the received record count.")

actual_keys = []
for record in list_trade:
    year = str(record.get("period"))
    if year not in YEARS:
        raise ValueError("Unexpected year in partner summary.")
    for field, expected in expected_fields.items():
        if record.get(field) != expected:
            raise ValueError(f"Unexpected value for {field}; review the response.")
    if not record.get("partnerCode") and record.get("partnerCode") != 0:
        raise ValueError("A record is missing its partner code.")
    if record.get("classificationCode") != EXPECTED_EDITION:
        raise ValueError("Unexpected HS edition; check comparability.")
    value = record.get("primaryValue")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError("A record is missing a numeric primary trade value.")
    if not math.isfinite(value) or value < 0:
        raise ValueError("A record has a non-finite or negative trade value.")
    actual_keys.append((year, str(record["partnerCode"])))

if len(actual_keys) != len(set(actual_keys)):
    raise ValueError("Duplicate year/partner records found.")

for year in YEARS:
    world_rows = [
        record for record in list_trade
        if str(record.get("period")) == year and str(record.get("partnerCode")) == "0"
    ]
    if len(world_rows) != 1:
        raise ValueError(f"Expected one World aggregate record for {year}.")

print(f"Validated {len(list_trade)} partner-total records across {len(YEARS)} years.")
print("The World aggregate is present once per year and must be excluded from partner rankings.")

Validated 673 partner-total records across 3 years.
The World aggregate is present once per year and must be excluded from partner rankings.


## Save the validated V2 partner-summary snapshot

In API mode, this cell saves the raw partner records, query scope, and request log in a new dated JSON file. It uses exclusive creation so it will not overwrite an existing snapshot, and stores no API key or request headers. Snapshot mode leaves the supplied file unchanged.

In [10]:
if DATA_SOURCE == "api":
    if len(extraction_log) != len(YEARS):
        raise ValueError("Complete all partner-summary requests before saving the snapshot.")
    if sum(entry["records_received"] for entry in extraction_log) != len(list_trade):
        raise ValueError("Extraction log does not match the collected partner records.")

    saved_at = datetime.now(timezone.utc)
    snapshot_data = {
        "schema_version": 2,
        "source": "UN Comtrade",
        "source_url": "https://comtradeplus.un.org/ApiDocs",
        "saved_at_utc": saved_at.isoformat(),
        "query_scope": query_scope,
        "extraction_log": extraction_log,
        "records": list_trade
    }
    snapshot_folder = Path("data")
    snapshot_folder.mkdir(exist_ok=True)
    snapshot_name = f"v2_partner_totals_{saved_at:%Y%m%dT%H%M%S%fZ}.json"
    saved_snapshot_path = snapshot_folder / snapshot_name

    with saved_snapshot_path.open("x", encoding="utf-8") as file:
        json.dump(snapshot_data, file, ensure_ascii=False, indent=2, allow_nan=False)

    print(f"Saved {len(list_trade)} partner-total records to {saved_snapshot_path}.")
else:
    print("Using the supplied V2 snapshot; no new snapshot was written.")

Using the supplied V2 snapshot; no new snapshot was written.


In [11]:
# Use Pandas to inspect the records; keep list_trade as the raw MongoDB input.
preview_columns = [
    "period", "reporterDesc", "partnerCode", "partnerDesc",
    "cmdCode", "cmdDesc", "classificationCode", "primaryValue"
]
df_preview = pd.DataFrame(list_trade)
df_preview.reindex(columns=preview_columns).head(20)

,period,reporterDesc,partnerCode,partnerDesc,cmdCode,cmdDesc,classificationCode,primaryValue
0,2023,Switzerland,0,World,TOTAL,All Commodities,H6,4.199225e+11
1,2023,Switzerland,4,Afghanistan,TOTAL,All Commodities,H6,5.718329e+06
2,2023,Switzerland,8,Albania,TOTAL,All Commodities,H6,6.241842e+07
3,2023,Switzerland,12,Algeria,TOTAL,All Commodities,H6,2.472354e+08
4,2023,Switzerland,16,American Samoa,TOTAL,All Commodities,H6,7.559643e+05
5,2023,Switzerland,20,Andorra,TOTAL,All Commodities,H6,5.274068e+06
6,2023,Switzerland,24,Angola,TOTAL,All Commodities,H6,1.126619e+07
7,2023,Switzerland,28,Antigua and Barbuda,TOTAL,All Commodities,H6,7.937302e+05
8,2023,Switzerland,31,Azerbaijan,TOTAL,All Commodities,H6,1.188964e+08
9,2023,Switzerland,32,Argentina,TOTAL,All Commodities,H6,7.615638e+08


## Understanding the V2 discovery data

### What one record represents

V2's first extract contains one annual all-commodity export total for each reported partner area. It includes World as an aggregate row; it does not yet contain HS4 product detail. `partnerCode=None` omits the request filter; each returned record has its own `partnerCode` and `partnerDesc`. The code `TOTAL` combines all commodities for discovery.

The number of partner records is determined from each API response; do not assume every possible area has a reported record in every year. Keep the World total for reconciliation but exclude it, and any overlapping aggregate areas, when ranking individual partner areas.

### Field guide: 47 source columns

| Group | Columns | How to read them |
|---|---|---|
| Type and time | `typeCode`, `freqCode`, `period`, `refYear`, `refMonth`, `refPeriodId` | `C` = goods; `A` = annual. `period` is the year as text. `refMonth` can be `52` for annual data, not an ordinary month number. |
| Reporter | `reporterCode`, `reporterISO`, `reporterDesc` | Switzerland is `757`, `CHE`, `Switzerland`. |
| Flow | `flowCode`, `flowDesc` | `X`, `Export`. |
| Partners | `partnerCode`, `partnerISO`, `partnerDesc`, `partner2Code`, `partner2ISO`, `partner2Desc` | Partner fields vary by row; `partnerCode = 0` is World. The additional partner dimension remains aggregated. |
| Classification and commodity | `classificationCode`, `classificationSearchCode`, `isOriginalClassification`, `cmdCode`, `cmdDesc`, `aggrLevel`, `isLeaf` | The requested classification is `HS`; the returned edition must be `H6`. In this discovery query `cmdCode` is `TOTAL`, not an HS4 product code. |
| Customs and modes | `customsCode`, `customsDesc`, `mosCode`, `motCode`, `motDesc` | Source dimensions for customs and transport/mode selections; this query uses aggregate selections. |
| Quantities and weights | `qtyUnitCode`, `qtyUnitAbbr`, `qty`, `isQtyEstimated`, `altQtyUnitCode`, `altQtyUnitAbbr`, `altQty`, `isAltQtyEstimated`, `netWgt`, `isNetWgtEstimated`, `grossWgt`, `isGrossWgtEstimated` | Measures and units; estimation flags show when the measurement is estimated. Do not assume a missing or estimated zero is a measured zero. |
| Trade values | `cifvalue`, `fobvalue`, `primaryValue` | `primaryValue` is the main current-US-dollar measure for ranking. Do not add these valuation fields together. |
| Status and provenance | `legacyEstimationFlag`, `isReported`, `isAggregate` | Source-status and estimation information; check Comtrade definitions before interpreting a code. |

### Later HS4 detail and MongoDB model

After the top 30 partner areas are selected in MongoDB, V2 will request their H6 HS4 records. A clean document will represent one **year × partner area × HS4 heading** observation. Reporter, partner, commodity, trade, and quality fields can be embedded together. The raw partner summary and later raw HS4 data should remain unchanged; the clean top-50 subset will be produced in MongoDB after ranking.

The HS hierarchy is chapter (2 digits) → heading (4 digits) → subheading (6 digits). The discovery query contains `TOTAL`, so it cannot itself answer product-level questions. Do not combine parent totals with their detailed child codes in one sum.

### HScode hierarchy example

`TOTAL` combines all commodities; it is not a numbered HS category. This example shows one branch, not product detail from the current dataset.

```mermaid
flowchart TD
    TOTAL["TOTAL: all commodities combined"]
    TOTAL --> SECTION["Section I: Live animals and animal products"]
    SECTION --> CHAPTER["01: Live animals"]
    CHAPTER --> HEADING["0101: Live horses, asses, mules and hinnies"]
    HEADING --> SUB1["010121: Purebred breeding horses"]
    HEADING --> SUB2["010129: Other horses"]
```

`H6` identifies HS 2022; HS4 identifies the four-digit heading level.

### Agreed clean-record shape: illustrative excerpt

The planned MongoDB transformation keeps one document per year, exporting country, destination area, and HS4 heading.
The raw collection keeps all source fields unchanged.

This excerpt uses the agreed field names, but does not show an existing clean record or every retained field.
Germany and heading `0101` illustrate the structure, not confirmed partner or heading selections.
The figures are illustrative, not downloaded results.
An estimated-weight flag does not establish that `0.0` represents a measured zero.

```json
{
  "year": "2023",
  "exportingCountry": {
    "comtradeCode": 757,
    "isoCode": "CHE",
    "name": "Switzerland"
  },
  "destinationArea": {
    "comtradeCode": 276,
    "isoCode": "DEU",
    "name": "Germany"
  },
  "tradeFlow": {
    "code": "X",
    "description": "Export"
  },
  "product": {
    "classificationEdition": "H6",
    "aggregationLevel": 4,
    "hsCode": "0101",
    "description": "Live horses, asses, mules and hinnies",
    "chapter": {
      "hsCode": "01"
    }
  },
  "tradeMeasures": {
    "primaryValueUSD": 12345.67,
    "fobValueUSD": 12345.67,
    "netWeightKg": 0.0
  },
  "dataQuality": {
    "isNetWeightEstimated": true
  }
}
```

```mermaid
flowchart TD
    RECORD["One clean observation"]
    RECORD --> YEAR["year: 2023"]
    RECORD --> REPORTER["exportingCountry: Switzerland"]
    RECORD --> PARTNER["destinationArea: Germany"]
    RECORD --> FLOW["tradeFlow: Export"]
    RECORD --> COMMODITY["product: HS4 heading 0101"]
    COMMODITY --> CHAPTER["chapter: HS2 code 01"]
    RECORD --> TRADE["tradeMeasures: values and weight"]
    RECORD --> QUALITY["dataQuality: source flags"]
```

- Unlike the current flat records, this model stores related fields inside actual nested objects.
- Embedding groups fields inside one observation; it does not place partner observations inside the World record.
- `product.chapter` stores only the parent code, not another observation or an additional value to sum.
- Derive the parent code from the heading prefix; use the saved H6 reference to label chapter results.
- Use existing HS4 observations for product values; do not split discovery `TOTAL` values into products.
- Keep commodity codes as strings to preserve leading zeros.

Python reads the destination name with `record["destinationArea"]["name"]`.
MongoDB uses the field path `destinationArea.name`.
The later model section gives the complete agreed mapping.

## Readable V2 partner-summary CSV

This CSV contains the partner discovery records and all source columns. It does not contain HS4 detail; that will be requested after the MongoDB stage selects the top 30 partners. Keep the JSON snapshot for reproducibility and load raw API records directly into MongoDB later, because CSV can lose types and leading zeros.

In [12]:
Path("data").mkdir(exist_ok=True)
trade_csv = Path("data") / f"v2_partner_totals_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.csv"
df_preview.to_csv(trade_csv, index=False, mode="x")
print(f"All {len(df_preview)} V2 partner-total records saved to {trade_csv}")

All 673 V2 partner-total records saved to data/v2_partner_totals_20261017T092929891101Z.csv


In [13]:
pd.DataFrame(extraction_log)[
    ["year_requested", "api_count", "records_received", "extracted_at_utc"]
]

,year_requested,api_count,records_received,extracted_at_utc
0,2023,226,226,2026-10-16T14:35:47.820549+00:00
1,2024,222,222,2026-10-16T14:35:54.531175+00:00
2,2025,225,225,2026-10-16T14:36:00.025211+00:00


## Database setup: connect to Atlas

The course uses `pymongo.MongoClient(CNX_STR)` to connect to MongoDB. We keep that pattern but read `MONGODB_URI` from `.env` or the environment instead of notebook source. Keep the URI private and preserve the existing Comtrade key in `.env`.

Run the imports cell, then set `CONNECT_TO_MONGODB = True` below to enable this check. You do not need to rerun the API extraction. The default `False` skips credentials and network access, including during snapshot runs.

`MongoClient` creates a client; `client.admin.command("ping")` verifies connectivity. `serverSelectionTimeoutMS=10000` limits server-selection waiting to ten seconds. These checks extend the course connection example. They do not insert, update, or delete data.

Course reference: [MongoDB Backends and Atlas](txt%20course%20content/Copy_paste_course_content.md).

In [ ]:
# %pip install -r requirements.txt
CONNECT_TO_MONGODB = True

Note: you may need to restart the kernel to use updated packages.


In [13]:
if CONNECT_TO_MONGODB:
    load_dotenv(dotenv_path=".env", override=False)
    CNX_STR = os.environ.get("MONGODB_URI", "").strip()
    if not CNX_STR:
        raise ValueError("Set MONGODB_URI in .env or your environment before connecting.")

    client = pymongo.MongoClient(CNX_STR, serverSelectionTimeoutMS=10000)
    client.admin.command("ping")
    print("MongoDB connection successful.")
else:
    print("MongoDB connection skipped. Set CONNECT_TO_MONGODB = True to test Atlas.")

MongoDB connection successful.


## V2 discovery interpretation and next steps

The discovery records contain annual all-commodity totals by partner area for 2023–2025. They are not the final HS4 dataset. Preserve this snapshot, then, after the MongoDB phase is authorized, load it into a raw collection and use an aggregation pipeline to rank partner areas by cumulative `primaryValue`.

Next, use the selected top-30 partner codes in a bounded H6 HS4 query. After loading those raw records, rank HS4 codes by value in MongoDB and create a clean top-50 collection. Keep World separate from partner rankings, do not fill missing combinations with zero, and document estimated or missing weights. V2 still needs the actual MongoDB model, analyses, diagrams, conclusions, and final PDF.

### Planned MongoDB workflow

This workflow describes future work, not completed MongoDB steps. Rank partners across 2023-2025, then rank headings across those selected partners and years.

```mermaid
flowchart TD
    A["Current partner-total snapshot"]
    A --> B["Connect to MongoDB Atlas and verify connection"]
    B --> C["Load partner_totals_raw"]
    C --> D["MongoDB: rank top 30 eligible partners"]
    D --> E["Download HS4 detail for those partners"]
    E --> F["Load hs4_raw"]
    F --> G["MongoDB: rank top 50 HS4 headings"]
    G --> H["Transform selected observations into trade_clean"]
    H --> I["Aggregation analysis and charts"]
```

### Select the database and collection

The course uses `db = client['db_test']` to select a database and `col = db.cl_test` to select a collection. We follow the same PyMongo pattern: `db` refers to `swiss_trade_v2`, and `collection_raw` refers to `partner_totals_raw`, where the original partner-total records will be stored before transformation.

These lines select database and collection handles; they do not insert or change documents. MongoDB creates them when the first documents are inserted. A count of zero means no documents were found yet.

In [14]:

DB_NAME = "swiss_trade_v2"
RAW_COLLECTION = "partner_totals_raw"

db = client[DB_NAME]
collection_raw = db[RAW_COLLECTION]

print("Database:", db.name)
print("Collection:", collection_raw.name)
print("Current documents:", collection_raw.count_documents({}))

Database: swiss_trade_v2
Collection: partner_totals_raw
Current documents: 673


In [17]:
# This code writes to Atlas.
if not list_trade:
    raise ValueError("Load and validate the snapshot first.")

for record in list_trade:
    document = record.copy()

    # Other query dimensions remain fixed in this collection.
    document["_id"] = f"{record['period']}_{record['partnerCode']}"

    collection_raw.replace_one(
        {"_id": document["_id"]},
        document,
        upsert=True
    )

print("Raw loading finished.")
print("Stored documents:", collection_raw.count_documents({}))

Raw loading finished.
Stored documents: 673


In [18]:
# Verify the stored count
for year in YEARS:
    count = collection_raw.count_documents({"period": year})
    print(year, count)

world_count = collection_raw.count_documents({"partnerCode": 0})
print("World records:", world_count)

2023 226
2024 222
2025 225
World records: 3


In [19]:
# Use MongoDB's aggregation pipeline, following the course's aggregate([...]) pattern.
# Sum three years of partner totals in MongoDB before requesting detailed HS4 data.
pipeline = [
    {
        # Keep Swiss export totals for the selected years; exclude World from partner ranking.
        "$match": {
            "period": {"$in": YEARS},
            "reporterCode": 757,
            "flowCode": "X",
            "cmdCode": "TOTAL",
            "partnerCode": {"$ne": 0}
        }
    },
    {
        # Group by partner, sum the yearly values, and count that partner's records.
        "$group": {
            "_id": "$partnerCode",
            "partnerName": {"$first": "$partnerDesc"},
            "totalExportValue": {"$sum": "$primaryValue"},
            "observations": {"$sum": 1}
        }
    },
    {
        # Sort largest totals first; partner code gives ties a consistent order.
        "$sort": {
            "totalExportValue": -1,
            "_id": 1
        }
    },
    {
        # Keep only the top 30 partners for the later HS4 extraction.
        "$limit": 30
    }
]

# MongoDB runs the pipeline; Python receives only the small ranked result.
top30_candidates = list(collection_raw.aggregate(pipeline))

pd.DataFrame(top30_candidates).rename(
    columns={"_id": "partnerCode"}
)
top30_table = pd.DataFrame(top30_candidates).rename(
    columns={"_id": "partnerCode"}
)

print(top30_table.to_string(index=False))

 partnerCode          partnerName  totalExportValue  observations
         842                  USA      2.571219e+11             3
         276              Germany      1.625739e+11             3
         156                China      1.229667e+11             3
         826       United Kingdom      9.536637e+10             3
         380                Italy      8.390953e+10             3
         705             Slovenia      7.879933e+10             3
         251               France      7.345312e+10             3
         699                India      5.535842e+10             3
         792              Türkiye      3.841508e+10             3
         344 China, Hong Kong SAR      3.507132e+10             3
         784 United Arab Emirates      3.380909e+10             3
          40              Austria      3.101915e+10             3
         392                Japan      2.737813e+10             3
         724                Spain      2.333638e+10             3
         7

In [20]:
selected_partner_codes = [
    row["_id"] for row in top30_candidates
]

if len(selected_partner_codes) != 30:
    raise ValueError("Expected 30 selected partners.")

if len(set(selected_partner_codes)) != 30:
    raise ValueError("Partner codes must be unique.")

if 0 in selected_partner_codes:
    raise ValueError("World must not appear in the partner selection.")

partner_codes_parameter = ",".join(
    str(code) for code in selected_partner_codes
)

print(partner_codes_parameter)

842,276,156,826,380,705,251,699,792,344,784,40,392,724,702,528,682,56,764,124,410,616,76,490,36,643,484,203,642,752


## Top 30 partner countries chosen from above

The scope includes the top 30 non-World partner areas by cumulative 2023-2025 export value. We retain code `490`, `Other Asia, nes`, with its official source label. The scope therefore covers partner areas, not only sovereign countries.

The HS4 snapshot will preserve the selected codes and ranking results. We will use the same partner selection for every year.

## Extract HS4 detail for the selected partners

`AG4` requests four-digit HS headings; it does not mean the top four products. We need all returned headings before MongoDB can rank the top 50. The response must use original H6 classification and aggregation level 4.

Keep the discovery `DATA_SOURCE` in snapshot mode. HS4 uses a separate `HS4_DATA_SOURCE`: `off`, `api`, or `snapshot`. The default `off` performs no extraction. Only `api` loads the Comtrade key and sends requests; `snapshot` reads a saved HS4 file without credentials or network access.

We request one partner and year per batch: 90 data requests before retries, plus one public H6 reference download. Successful data batches have at least five seconds between them. Check your remaining quota before starting. The shared helper checks response counts without truncating fractional values.

Run the updated download-function cell before this section. Never print request headers or credentials. An empty batch means the API returned no rows; it does not establish zero exports.

In [9]:
HS4_DATA_SOURCE = "off"  # off, api, or snapshot
HS4_SNAPSHOT_PATH = Path("data/v2_hs4_snapshot_20261017T094752738553Z.json")
HS4_RAW_COLLECTION = "hs4_raw"
HS4_REFERENCE_URL = "https://comtradeapi.un.org/files/v1/app/reference/H6.json"
LOAD_HS4_TO_MONGODB = False
RANK_HS4_IN_MONGODB = False

hs4_base_params = base_params.copy()
hs4_base_params["cmdCode"] = "AG4"
hs4_base_params.pop("partnerCode", None)

if HS4_DATA_SOURCE not in {"off", "api", "snapshot"}:
    raise ValueError("Choose off, api, or snapshot for HS4_DATA_SOURCE.")

In [10]:
def make_hs4_scope(partner_codes):
    if not isinstance(partner_codes, list) or len(partner_codes) != 30:
        raise ValueError("HS4 extraction requires 30 selected partner codes.")
    if any(isinstance(code, bool) or not isinstance(code, int) or code <= 0 for code in partner_codes):
        raise ValueError("HS4 partner codes must be positive integers; exclude World.")
    if len(set(partner_codes)) != 30:
        raise ValueError("HS4 partner codes must be unique.")
    return {
        "endpoint": API_URL,
        "years": YEARS.copy(),
        "filters": hs4_base_params.copy(),
        "expected_edition": EXPECTED_EDITION,
        "selected_partner_codes": partner_codes.copy()
    }


def validate_hs4_batch(records, year, partner_code):
    if not isinstance(records, list):
        raise ValueError("Expected a list of HS4 records.")
    expected = {
        "period": year, "partnerCode": partner_code,
        "reporterCode": hs4_base_params["reporterCode"],
        "typeCode": "C", "freqCode": "A", "aggrLevel": 4,
        "classificationCode": EXPECTED_EDITION, "isOriginalClassification": True,
        "flowCode": hs4_base_params["flowCode"],
        "partner2Code": hs4_base_params["partner2Code"],
        "customsCode": hs4_base_params["customsCode"],
        "motCode": hs4_base_params["motCode"]
    }
    codes = []
    for record in records:
        if not isinstance(record, dict):
            raise ValueError("Each HS4 record must be an object.")
        for field, value in expected.items():
            if record.get(field) != value:
                raise ValueError(f"Unexpected HS4 field {field} for {year}, partner {partner_code}.")
        code = record.get("cmdCode")
        if not isinstance(code, str) or len(code) != 4 or not code.isascii() or not code.isdigit():
            raise ValueError("Expected a four-digit HS4 code, not TOTAL or another level.")
        value = record.get("primaryValue")
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError("Each HS4 record needs a numeric primaryValue.")
        if not math.isfinite(value) or value < 0:
            raise ValueError("HS4 primaryValue must be finite and nonnegative.")
        codes.append(code)
    if len(codes) != len(set(codes)):
        raise ValueError("Duplicate year/partner/HS4 records found.")


def validate_hs4_extract(records, extraction_log, scope):
    if not isinstance(scope, dict) or scope != make_hs4_scope(scope.get("selected_partner_codes")):
        raise ValueError("HS4 snapshot query scope differs from the notebook settings.")
    if not isinstance(records, list) or not records:
        raise ValueError("No HS4 records are available.")
    if not isinstance(extraction_log, list):
        raise ValueError("Expected an HS4 extraction log.")
    batches = {
        (year, partner): []
        for year in scope["years"] for partner in scope["selected_partner_codes"]
    }
    for record in records:
        if not isinstance(record, dict):
            raise ValueError("Each HS4 record must be an object.")
        key = (record.get("period"), record.get("partnerCode"))
        if key not in batches:
            raise ValueError("HS4 record falls outside the selected years or partners.")
        batches[key].append(record)
    seen = set()
    for entry in extraction_log:
        if not isinstance(entry, dict):
            raise ValueError("Each HS4 extraction-log entry must be an object.")
        key = (entry.get("year_requested"), entry.get("partner_requested"))
        if key not in batches or key in seen:
            raise ValueError("HS4 extraction log has an unexpected or repeated batch.")
        seen.add(key)
        count = entry.get("api_count")
        if isinstance(count, bool) or not isinstance(count, int) or not 0 <= count < MAX_RECORDS:
            raise ValueError("Invalid HS4 response count or record cap reached.")
        if count != entry.get("records_received") or count != len(batches[key]):
            raise ValueError("HS4 batch counts do not match the records.")
        expected_query = {**hs4_base_params, "period": key[0], "partnerCode": key[1]}
        if entry.get("query") != expected_query or entry.get("endpoint") != API_URL:
            raise ValueError("HS4 batch query does not match the requested scope.")
        validate_hs4_batch(batches[key], key[0], key[1])
    if seen != set(batches):
        raise ValueError("HS4 extraction log must cover all 90 year/partner batches.")

### Reference codes and value coverage

The H6 reference identifies standard headings; four digits alone do not prove that a code is a valid heading. API mode downloads this public reference once. The HS4 snapshot preserves it, so later snapshot runs need no reference download.

We preserve nonstandard source codes in the raw data and list them separately. The top-50 pipeline includes only standard H6 headings. Reference chapters 00, 77, 98, and 99 do not represent standard international HS chapters.

A Pandas quality check compares each partner-year's HS4 value sum with its frozen all-commodity total. It also reports coverage from standard headings alone. Differences need investigation; they do not justify automatic rescaling or inventing missing observations.

In [11]:
def get_hs4_heading_codes(reference):
    if not isinstance(reference, dict) or reference.get("source_url") != HS4_REFERENCE_URL:
        raise ValueError("Expected the saved H6 reference and its source URL.")
    payload = reference.get("data")
    if not isinstance(payload, dict) or not isinstance(payload.get("results"), list):
        raise ValueError("Expected a results list in the H6 reference.")
    heading_codes = set()
    for item in payload["results"]:
        if not isinstance(item, dict):
            raise ValueError("Each H6 reference entry must be an object.")
        code = item.get("id")
        if isinstance(code, str) and len(code) == 4 and code.isascii() and code.isdigit():
            chapter = int(code[:2])
            if 1 <= chapter <= 97 and chapter != 77:
                heading_codes.add(code)
    if not heading_codes:
        raise ValueError("The H6 reference contains no standard four-digit headings.")
    return heading_codes


def validate_hs4_control_totals(control_totals, scope):
    if not isinstance(control_totals, list):
        raise ValueError("Expected saved partner-year control totals.")
    expected_keys = {
        (year, partner)
        for year in scope["years"] for partner in scope["selected_partner_codes"]
    }
    seen = set()
    for record in control_totals:
        if not isinstance(record, dict):
            raise ValueError("Each control total must be an object.")
        key = (record.get("period"), record.get("partnerCode"))
        if key not in expected_keys or key in seen:
            raise ValueError("Unexpected or duplicate partner-year control total.")
        value = record.get("primaryValue")
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError("Control totals must contain numeric trade values.")
        if not math.isfinite(value) or value < 0:
            raise ValueError("Control totals must be finite and nonnegative.")
        seen.add(key)
    if seen != expected_keys:
        raise ValueError("Control totals must cover every selected partner and year.")


def hs4_coverage_table(records, control_totals, scope, heading_codes):
    validate_hs4_control_totals(control_totals, scope)
    keys = ["period", "partnerCode"]
    details = pd.DataFrame(records, columns=keys + ["cmdCode", "primaryValue"])
    totals = pd.DataFrame(control_totals)[keys + ["primaryValue"]]
    totals = totals.rename(columns={"primaryValue": "totalValue"})
    raw_sums = details.groupby(keys, as_index=False)["primaryValue"].sum()
    raw_sums = raw_sums.rename(columns={"primaryValue": "hs4Value"})
    standard_rows = details[details["cmdCode"].isin(heading_codes)]
    standard_sums = standard_rows.groupby(keys, as_index=False)["primaryValue"].sum()
    standard_sums = standard_sums.rename(columns={"primaryValue": "standardHeadingValue"})
    coverage = totals.merge(raw_sums, on=keys, how="left", validate="one_to_one")
    coverage = coverage.merge(standard_sums, on=keys, how="left", validate="one_to_one")
    coverage["differenceUSD"] = coverage["hs4Value"] - coverage["totalValue"]
    denominator = coverage["totalValue"].where(coverage["totalValue"] != 0)
    coverage["coveragePct"] = 100 * coverage["hs4Value"] / denominator
    coverage["standardCoveragePct"] = 100 * coverage["standardHeadingValue"] / denominator
    coverage["absoluteDifferenceUSD"] = coverage["differenceUSD"].abs()
    return coverage

### Run extraction or reproduce a snapshot

For the first download, set `HS4_DATA_SOURCE = "api"` only in the settings cell. Keep the extraction cell's `if` and `elif` conditions unchanged. Run the settings, validation-functions, reference-and-coverage-functions, and extraction cells in order. The notebook loads the API key separately from the discovery mode. Existing partner totals and their settings remain unchanged.

For reproduction, set `HS4_DATA_SOURCE = "snapshot"` and provide the exact saved filename. Snapshot mode restores the frozen partner selection and checks it against the current years, filters, and edition. It does not rerun the partner ranking or require Atlas for these HS4 cells.

In [26]:
hs4_records = []
hs4_extraction_log = []
hs4_query_scope = None
hs4_partner_ranking = []
hs4_reference = None
hs4_control_totals = []
hs4_heading_codes = set()
hs4_nonstandard_codes = []

if HS4_DATA_SOURCE == "snapshot":
    with HS4_SNAPSHOT_PATH.open(encoding="utf-8") as file:
        hs4_snapshot = json.load(file)
    if not isinstance(hs4_snapshot, dict) or hs4_snapshot.get("dataset") != "swiss_exports_hs4":
        raise ValueError("Expected an HS4 snapshot, not the partner-total snapshot.")
    if hs4_snapshot.get("schema_version") != 2:
        raise ValueError("Unsupported HS4 snapshot version.")
    hs4_query_scope = hs4_snapshot.get("query_scope")
    hs4_records = hs4_snapshot.get("records")
    hs4_extraction_log = hs4_snapshot.get("extraction_log")
    hs4_partner_ranking = hs4_snapshot.get("partner_ranking")
    hs4_reference = hs4_snapshot.get("heading_reference")
    hs4_control_totals = hs4_snapshot.get("control_totals")
elif HS4_DATA_SOURCE == "api":
    hs4_query_scope = make_hs4_scope(selected_partner_codes)
    hs4_partner_ranking = [row.copy() for row in top30_candidates]
    if [row["_id"] for row in hs4_partner_ranking] != selected_partner_codes:
        raise ValueError("Selected partner codes differ from the MongoDB ranking.")
    for record in list_trade:
        if record["period"] in YEARS and record["partnerCode"] in selected_partner_codes:
            hs4_control_totals.append({
                "period": record["period"], "partnerCode": record["partnerCode"],
                "primaryValue": record["primaryValue"]
            })
    validate_hs4_control_totals(hs4_control_totals, hs4_query_scope)
    load_dotenv(dotenv_path=".env", override=False)
    hs4_api_key = os.environ.get("UN_COMTRADE_API_KEY", "").strip()
    if not hs4_api_key:
        hs4_api_key = getpass("UN Comtrade API key for HS4 extraction: ").strip()
    if not hs4_api_key:
        raise ValueError("An API key is required for HS4 extraction.")
    hs4_headers = {"Ocp-Apim-Subscription-Key": hs4_api_key}
    reference_response = requests.get(HS4_REFERENCE_URL, timeout=45)
    reference_response.raise_for_status()
    hs4_reference = {
        "source_url": HS4_REFERENCE_URL,
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "data": reference_response.json()
    }
    get_hs4_heading_codes(hs4_reference)
    batch_number = 0
    batch_total = len(YEARS) * len(selected_partner_codes)
    for year in YEARS:
        for partner_code in selected_partner_codes:
            batch_number += 1
            print(f"HS4 batch {batch_number}/{batch_total}: {year}, partner {partner_code}")
            params = {**hs4_base_params, "partnerCode": partner_code}
            extracted_at = datetime.now(timezone.utc).isoformat()
            records, api_count = request_comtrade_records(year, params, hs4_headers)
            validate_hs4_batch(records, year, partner_code)
            hs4_records.extend(records)
            hs4_extraction_log.append({
                "year_requested": year, "partner_requested": partner_code,
                "api_count": api_count, "records_received": len(records),
                "extracted_at_utc": extracted_at, "endpoint": API_URL,
                "query": {**params, "period": year}
            })
            if batch_number < batch_total:
                time.sleep(max(5, REQUEST_DELAY))
else:
    print("HS4 extraction skipped. Choose api or snapshot when ready.")

if HS4_DATA_SOURCE in {"api", "snapshot"}:
    validate_hs4_extract(hs4_records, hs4_extraction_log, hs4_query_scope)
    if not isinstance(hs4_partner_ranking, list) or any(not isinstance(row, dict) for row in hs4_partner_ranking):
        raise ValueError("HS4 snapshot must preserve the partner ranking.")
    if [row.get("_id") for row in hs4_partner_ranking] != hs4_query_scope["selected_partner_codes"]:
        raise ValueError("HS4 snapshot partner ranking differs from its selected codes.")
    hs4_heading_codes = get_hs4_heading_codes(hs4_reference)
    validate_hs4_control_totals(hs4_control_totals, hs4_query_scope)
    source_codes = {record["cmdCode"] for record in hs4_records}
    hs4_nonstandard_codes = sorted(source_codes - hs4_heading_codes)
    print(f"Validated {len(hs4_records)} HS4 records from {len(hs4_extraction_log)} batches.")
    print("Nonstandard source codes:", hs4_nonstandard_codes)

HS4 batch 1/90: 2023, partner 842
HS4 batch 2/90: 2023, partner 276
HS4 batch 3/90: 2023, partner 156
HS4 batch 4/90: 2023, partner 826
HS4 batch 5/90: 2023, partner 380
HS4 batch 6/90: 2023, partner 705
HS4 batch 7/90: 2023, partner 251
HS4 batch 8/90: 2023, partner 699
HS4 batch 9/90: 2023, partner 792
HS4 batch 10/90: 2023, partner 344
HS4 batch 11/90: 2023, partner 784
HS4 batch 12/90: 2023, partner 40
HS4 batch 13/90: 2023, partner 392
HS4 batch 14/90: 2023, partner 724
HS4 batch 15/90: 2023, partner 702
HS4 batch 16/90: 2023, partner 528
HS4 batch 17/90: 2023, partner 682
HS4 batch 18/90: 2023, partner 56
HS4 batch 19/90: 2023, partner 764
HS4 batch 20/90: 2023, partner 124
HS4 batch 21/90: 2023, partner 410
HS4 batch 22/90: 2023, partner 616
HS4 batch 23/90: 2023, partner 76
HS4 batch 24/90: 2023, partner 490
HS4 batch 25/90: 2023, partner 36
HS4 batch 26/90: 2023, partner 643
HS4 batch 27/90: 2023, partner 484
HS4 batch 28/90: 2023, partner 203
HS4 batch 29/90: 2023, partner 64

### Preserve the validated HS4 extraction

Save only after all 90 batches pass validation. Snapshot schema version 2 preserves raw records, logs, query settings, partner ranking, H6 reference, and partner-year control totals. It contains no request headers or API keys. Exclusive file creation prevents overwriting an existing snapshot.

Copy the printed filename into `HS4_SNAPSHOT_PATH` before switching to snapshot mode. Missing partner-product-year combinations remain absent; we do not manufacture zero-valued observations.

In [27]:
if HS4_DATA_SOURCE == "api":
    validate_hs4_extract(hs4_records, hs4_extraction_log, hs4_query_scope)
    get_hs4_heading_codes(hs4_reference)
    validate_hs4_control_totals(hs4_control_totals, hs4_query_scope)
    saved_at = datetime.now(timezone.utc)
    hs4_snapshot_data = {
        "schema_version": 2, "dataset": "swiss_exports_hs4",
        "source": "UN Comtrade", "source_url": "https://comtradeplus.un.org/ApiDocs",
        "saved_at_utc": saved_at.isoformat(),
        "query_scope": hs4_query_scope, "partner_ranking": hs4_partner_ranking,
        "heading_reference": hs4_reference, "control_totals": hs4_control_totals,
        "extraction_log": hs4_extraction_log, "records": hs4_records
    }
    Path("data").mkdir(exist_ok=True)
    hs4_saved_snapshot_path = Path("data") / f"v2_hs4_snapshot_{saved_at:%Y%m%dT%H%M%S%fZ}.json"
    with hs4_saved_snapshot_path.open("x", encoding="utf-8") as file:
        json.dump(hs4_snapshot_data, file, ensure_ascii=False, indent=2, allow_nan=False)
    print(f"Saved {len(hs4_records)} HS4 records to {hs4_saved_snapshot_path}.")
elif HS4_DATA_SOURCE == "snapshot":
    print("Using the supplied HS4 snapshot; no new snapshot was written.")
else:
    print("HS4 snapshot saving skipped.")

Saved 72459 HS4 records to data/v2_hs4_snapshot_20261017T094752738553Z.json.


In [28]:
if HS4_DATA_SOURCE in {"api", "snapshot"}:
    hs4_preview = pd.DataFrame(hs4_records)
    print(hs4_preview[["period", "partnerCode", "partnerDesc", "cmdCode", "cmdDesc", "primaryValue"]].head(10).to_string(index=False))
    hs4_log_table = pd.DataFrame(hs4_extraction_log)
    print(hs4_log_table.groupby("year_requested")["records_received"].sum().to_string())
    print("Empty batches:", (hs4_log_table["records_received"] == 0).sum())

period  partnerCode partnerDesc cmdCode                                                                                                                                                                                                                                     cmdDesc  primaryValue
  2023          842         USA    0101                                                                                                                                                                                                      Horses, asses, mules and hinnies; live  6.499938e+06
  2023          842         USA    0106                                                                                                                                                                                                         Animals; live, n.e.c. in chapter 01  2.657290e+05
  2023          842         USA    0210                                                                                           

### Compare HS4 coverage with partner totals

The table compares all returned four-digit source rows and standard H6 headings with the discovery totals. `differenceUSD` means the source-row sum minus the all-commodity total. `coveragePct` and `standardCoveragePct` report the two coverage percentages.

Review the largest differences before interpreting rankings or shares. Missing HS4 batches retain missing coverage, not zero coverage. A zero control total has no defined coverage percentage. We do not rescale values or require exact equality, because source rounding and reporting coverage can differ.

In [29]:
if HS4_DATA_SOURCE in {"api", "snapshot"}:
    validate_hs4_extract(hs4_records, hs4_extraction_log, hs4_query_scope)
    hs4_heading_codes = get_hs4_heading_codes(hs4_reference)
    hs4_coverage = hs4_coverage_table(
        hs4_records, hs4_control_totals, hs4_query_scope, hs4_heading_codes
    )
    largest_gaps = hs4_coverage.sort_values("absoluteDifferenceUSD", ascending=False)
    print(largest_gaps.head(10).to_string(index=False))
    print("Partner-years without HS4 rows:", hs4_coverage["hs4Value"].isna().sum())
    hs4_nonstandard_codes = sorted({row["cmdCode"] for row in hs4_records} - hs4_heading_codes)
    print("Nonstandard source codes excluded from heading ranking:", hs4_nonstandard_codes)

period  partnerCode   totalValue     hs4Value  standardHeadingValue  differenceUSD  coveragePct  standardCoveragePct  absoluteDifferenceUSD
  2024          764 4.954415e+09 4.954415e+09          4.949646e+09       0.018001        100.0            99.903735               0.018001
  2024          702 6.384549e+09 6.384549e+09          6.359975e+09       0.017000        100.0            99.615111               0.017000
  2023          642 1.805641e+09 1.805641e+09          1.801693e+09       0.016000        100.0            99.781363               0.016000
  2025          203 2.597759e+09 2.597759e+09          2.596291e+09       0.016000        100.0            99.943500               0.016000
  2024          203 2.331849e+09 2.331849e+09          2.330070e+09       0.015000        100.0            99.923727               0.015000
  2025          380 2.919904e+10 2.919904e+10          2.918047e+10       0.014999        100.0            99.936394               0.014999
  2024          826 

## Load HS4 records and rank headings in MongoDB

Only enable `LOAD_HS4_TO_MONGODB` after saving the snapshot and reviewing coverage. The loader keeps the course's replacement/upsert approach. It uses PyMongo `ReplaceOne` operations with `bulk_write()` in batches of at most 1,000. This small extension reduces Atlas round trips without changing document contents or rerun behavior.

Reruns replace matching observations instead of duplicating them. The loader refuses stale observations absent from the current snapshot before writing. Use a new collection for incompatible snapshots; the notebook never deletes existing records automatically.

`RANK_HS4_IN_MONGODB` enables a separate read-only pipeline. The check first verifies that stored source fields match the current snapshot. `$match` includes only headings from the saved H6 reference; nonstandard source rows remain in `hs4_raw`. `$group`, `$sum`, `$sort`, and `$limit` rank 50 headings across all selected partners and years, not 50 per partner.

Course references: [CRUD and aggregation examples](txt%20course%20content/Copy_paste_course_content.md). Pandas displays the returned results; MongoDB performs the ranking.

In [12]:
def hs4_document_id(record):
    fields = [
        "reporterCode", "period", "partnerCode", "partner2Code", "cmdCode",
        "classificationCode", "flowCode", "customsCode", "motCode"
    ]
    return "_".join(str(record[field]) for field in fields)


def hs4_mongodb_filter(scope):
    filters = scope["filters"]
    return {
        "period": {"$in": scope["years"]},
        "partnerCode": {"$in": scope["selected_partner_codes"]},
        "reporterCode": filters["reporterCode"],
        "flowCode": filters["flowCode"],
        "partner2Code": filters["partner2Code"],
        "customsCode": filters["customsCode"], "motCode": filters["motCode"],
        "typeCode": "C", "freqCode": "A", "aggrLevel": 4,
        "classificationCode": scope["expected_edition"],
        "isOriginalClassification": True,
        "cmdCode": {"$regex": "^[0-9]{4}$"}
    }


def verify_hs4_collection(collection, records, scope):
    expected = {hs4_document_id(record): record for record in records}
    seen = set()
    for document in collection.find(hs4_mongodb_filter(scope)):
        source_record = document.copy()
        document_id = source_record.pop("_id")
        if document_id not in expected or source_record != expected[document_id]:
            raise ValueError("Stored HS4 records differ from the snapshot; check the raw load.")
        seen.add(document_id)
    if seen != set(expected):
        raise ValueError("Stored HS4 records are incomplete; run the HS4 load first.")
    return len(seen)


def load_hs4_records(collection, records, extraction_log, scope):
    validate_hs4_extract(records, extraction_log, scope)
    expected_ids = {hs4_document_id(record) for record in records}
    stored_ids = {
        document["_id"]
        for document in collection.find(hs4_mongodb_filter(scope), {"_id": 1})
    }
    if stored_ids - expected_ids:
        raise ValueError("Existing HS4 scope contains stale records. Use a new collection; no automatic deletion.")
    operations = []
    for record in records:
        document = record.copy()
        document["_id"] = hs4_document_id(record)
        operations.append(pymongo.ReplaceOne({"_id": document["_id"]}, document, upsert=True))
        if len(operations) == 1000:
            collection.bulk_write(operations, ordered=False)
            operations = []
    if operations:
        collection.bulk_write(operations, ordered=False)
    return verify_hs4_collection(collection, records, scope)

In [36]:
if LOAD_HS4_TO_MONGODB:
    if not CONNECT_TO_MONGODB:
        raise ValueError("Enable and verify the Atlas connection before loading HS4 records.")
    if HS4_RAW_COLLECTION == RAW_COLLECTION:
        raise ValueError("Use a separate collection for HS4 detail, not partner totals.")
    collection_hs4_raw = db[HS4_RAW_COLLECTION]
    hs4_stored_count = load_hs4_records(
        collection_hs4_raw, hs4_records, hs4_extraction_log, hs4_query_scope
    )
    print(f"Verified {hs4_stored_count} stored HS4 documents in {HS4_RAW_COLLECTION}.")
else:
    print("HS4 MongoDB loading skipped. Enable LOAD_HS4_TO_MONGODB when ready.")

Verified 72459 stored HS4 documents in hs4_raw.


### HS4 raw documents in Atlas

![Atlas shows the HS4 raw collection and a 2023 Swiss export record for the USA.](screenshots/Screenshot%202026-10-17%20at%2011.58.15.png)

Atlas shows 72,459 documents in `swiss_trade_v2.hs4_raw`, matching the verified raw load. Each document keeps 47 source fields plus our `_id`. These are raw observations, not the final top-50 subset or the planned nested clean model.
each document represents one specific trade observation.

### Read the document identifier

The displayed `_id` is a string that our loading function creates:

```text
757_2023_842_0_0101_H6_X_C00_0
```

| Part | Source field | Meaning |
|---|---|---|
| `757` | `reporterCode` | Switzerland reports the trade |
| `2023` | `period` | Annual observation for 2023 |
| `842` | `partnerCode` | USA is the primary partner |
| `0` | `partner2Code` | Aggregate second-partner dimension |
| `0101` | `cmdCode` | Live horses, asses, mules and hinnies |
| `H6` | `classificationCode` | HS 2022 edition |
| `X` | `flowCode` | Exports |
| `C00` | `customsCode` | Aggregate customs selection |
| `0` | `motCode` | Aggregate transport selection |

MongoDB requires unique `_id` values within a collection. Our identifier differs from MongoDB's automatically generated `ObjectId`. Reruns use the same identifier to replace the same observation rather than create a duplicate.

Changing the year, partner, heading, or another identifier dimension produces a different `_id`. The HS code alone does not uniquely identify a trade observation.

### Read the fields visible in the screenshot

The types below use Python terminology. JSON groups integers and floating-point values under its number type.

| Field or related fields | Type | Displayed value | Meaning |
|---|---|---|---|
| `_id` | string | Identifier above | Unique observation key within this collection |
| `typeCode` | string | `C` | Goods rather than services |
| `freqCode` | string | `A` | Annual rather than monthly data |
| `refPeriodId` | integer | `20230101` | Source period identifier, not a shipment date |
| `refYear` | integer | `2023` | Numeric reference year |
| `refMonth` | integer | `52` | Source marker for annual data, not a calendar month |
| `period` | string | `2023` | Year field used in our queries |
| `reporterCode` | integer | `757` | Comtrade code for the reporting country |
| `reporterISO`, `reporterDesc` | strings | `CHE`, `Switzerland` | Reporter ISO code and name |
| `flowCode`, `flowDesc` | strings | `X`, `Export` | Trade direction and description |
| `partnerCode` | integer | `842` | Comtrade code for the primary partner |
| `partnerISO`, `partnerDesc` | strings | `USA`, `USA` | Partner ISO code and source label |
| `partner2Code` | integer | `0` | Aggregate second-partner selection |
| `partner2ISO`, `partner2Desc` | strings | `W00`, `World` | Labels for that additional aggregate dimension |
| `classificationCode` | string | `H6` | HS 2022 edition, not a six-digit product level |
| `classificationSearchCode` | string | `HS` | Requested HS classification family |
| `isOriginalClassification` | boolean | `true` | Source uses its original classification rather than a converted edition |
| `cmdCode` | string | `0101` | Four-digit HS heading; preserve the leading zero |
| `cmdDesc` | string | `Horses, asses, mules and hinnies; live` | Description of that heading |
| `aggrLevel` | integer | `4` | Four-digit heading aggregation level |

`partner2Desc = World` does not change the primary destination from USA to World. It describes the separate second-partner dimension, which we keep aggregated.

### Measures and flags below the visible area

The saved HS4 snapshot contains these additional values for the same observation:

| Field or related fields | Example value | Meaning |
|---|---|---|
| `primaryValue`, `fobvalue` | `6499937.824` each | Annual trade value in current US dollars; do not add these two fields together |
| `qty`, `qtyUnitAbbr` | `46.0`, `u` | Source quantity of 46 units, not 46 MongoDB documents |
| `netWgt`, `isNetWgtEstimated` | `23000.0`, `false` | Net weight in kilograms; the source does not mark it as estimated |
| `cifvalue` | `null` | No CIF value in this record; null does not mean zero |
| `isLeaf`, `isAggregate`, `isReported` | `false`, `true`, `false` | Source hierarchy and reporting flags for this heading-level total |

`isReported=false` does not mean missing or failed data; Comtrade can construct heading totals from finer detail. This HS4 record contains a positive weight; earlier discovery totals had unusable placeholder weights. Check units and estimation flags per record before analyzing physical quantities.

In [40]:
top50_hs4 = []
top50_hs4_codes = []
if RANK_HS4_IN_MONGODB:
    if not CONNECT_TO_MONGODB:
        raise ValueError("Enable and verify the Atlas connection before ranking HS4 records.")
    if HS4_RAW_COLLECTION == RAW_COLLECTION:
        raise ValueError("Rank HS4 detail in its separate raw collection.")
    validate_hs4_extract(hs4_records, hs4_extraction_log, hs4_query_scope)
    collection_hs4_raw = db[HS4_RAW_COLLECTION]
    verify_hs4_collection(collection_hs4_raw, hs4_records, hs4_query_scope)
    hs4_heading_codes = get_hs4_heading_codes(hs4_reference)
    validate_hs4_control_totals(hs4_control_totals, hs4_query_scope)
    ranking_filter = hs4_mongodb_filter(hs4_query_scope)
    ranking_filter["cmdCode"] = {"$in": sorted(hs4_heading_codes)}
    hs4_pipeline = [
        {"$match": ranking_filter},
        {"$group": {
            "_id": "$cmdCode",
            "commodityName": {"$first": "$cmdDesc"},
            "totalExportValue": {"$sum": "$primaryValue"},
            "observations": {"$sum": 1}
        }},
        {"$sort": {"totalExportValue": -1, "_id": 1}},
        {"$limit": 50}
    ]
    top50_hs4 = list(collection_hs4_raw.aggregate(hs4_pipeline))
    if len(top50_hs4) != 50:
        raise ValueError("Fewer than 50 HS4 headings are available; review the extraction scope.")
    top50_hs4_codes = [row["_id"] for row in top50_hs4]
    print(pd.DataFrame(top50_hs4).rename(columns={"_id": "hs4Code"}).to_string(index=False))
else:
    print("HS4 ranking skipped. Enable RANK_HS4_IN_MONGODB after verifying the raw load.")

hs4Code                                                                                                                                                                                                                                                  commodityName  totalExportValue  observations
   7108                                                                                                                                                          Gold (including gold plated with platinum) unwrought or in semi-manufactured forms, or in powder form      4.082109e+11            84
   3002  Human blood; animal blood for therapeutic, prophylactic or diagnostic uses; antisera, other blood fractions, immunological products, modified or obtained by biotechnological processes; vaccines, toxins, cultures of micro-organisms (excluding yeasts) etc      1.633355e+11            90
   3004             Medicaments; (not goods of heading no. 3002, 3005 or 3006) consisting of mixed or unmixed produ

### Next stage: build the nested clean collection

The top 50 headings cover 85.72% of standard-heading export value within the selected 30 partner areas across 2023-2025. This percentage does not describe all Swiss exports. 3 years × 30 partners = 90 possible observations per HS4 code 90 × 50 headings = 4,500 possible observations

`observations` counts year-partner records, not shipments. Each heading can have at most 90 observations: 30 partners across three years. Electricity has 12 observations covering four partners across those years.

The selected subset contains **4,235 observations** out of 4,500 possible combinations. 4,500 represents all possible combinations—not combinations that must exist.
We selected the top 50 headings overall, not headings that every partner imports from Switzerland.

```mermaid
pie showData
    title Selected year-partner-heading combinations
    "Existing observations" : 4235
    "Absent combinations" : 265
```

The main clean collection will keep all standard HS4 observations, not only this top-50 subset.

```mermaid
flowchart TD
    RAW["hs4_raw: 72,459 flat documents"]
    RAW --> MATCH["Keep all standard H6 headings: 72,369 observations"]
    MATCH --> SHAPE["MongoDB transformation: nest related fields"]
    SHAPE --> CLEAN["Planned trade_clean: 72,369 nested documents"]
```

- Keep one document per year, partner, and HS4 heading, with the other reporting dimensions fixed.
- Group fields into `reporter`, `partner`, `flow`, `commodity`, `trade`, and `quality` objects.
- Preserve both raw collections and do not add artificial zero-valued observations.
- Check the destination before writing; `$out` can replace an existing collection.

Expected clean counts are 23,805 for 2023, 24,188 for 2024, and 24,376 for 2025.
The 90 nonstandard `9999` rows remain raw.
Derive HS2 totals by grouping all standard HS4 observations under their parent chapter codes.
Keep the 4,235-observation top-50 subset as an optional drill-down.

## Data quality checks before transformation

These checks read `hs4_raw` without changing documents or creating collections.
They use the frozen HS4 snapshot and its H6 reference, not a new download or partner ranking.
MongoDB calculates the counts and summaries; Pandas only displays returned results.

Use the existing connected session, or run the imports and Atlas connection cells first.
Also define the existing query settings, HS4 settings, validation functions, reference functions, and MongoDB helper functions.
Those definition cells do not download or write data.
**Do not run all cells:** review earlier API and write settings before execution.
Set `CHECK_HS4_DATA_QUALITY = True` below, then run this section in order.

Each pipeline uses named stages and `collection.aggregate(pipeline)`, following the course examples.
The aggregation expression `$type` returns BSON type names, including `missing` and `null`.
This distinguishes absent fields from fields containing an explicit null value.

In [15]:
from pathlib import Path

CHECK_HS4_DATA_QUALITY = True
QUALITY_SNAPSHOT_PATH = Path("data/v2_hs4_snapshot_20261017T094752738553Z.json")

if CHECK_HS4_DATA_QUALITY:
    if not CONNECT_TO_MONGODB:
        raise ValueError("Connect to Atlas before running data quality checks.")
    if db.name != "swiss_trade_v2" or HS4_RAW_COLLECTION != "hs4_raw":
        raise ValueError("Use swiss_trade_v2.hs4_raw for these checks.")
    if HS4_RAW_COLLECTION not in db.list_collection_names():
        raise ValueError("The HS4 raw collection does not exist.")
    quality_snapshot = json.loads(QUALITY_SNAPSHOT_PATH.read_text(encoding="utf-8"))
    quality_scope = quality_snapshot["query_scope"]
    validate_hs4_extract(
        quality_snapshot["records"], quality_snapshot["extraction_log"], quality_scope
    )
    quality_heading_codes = get_hs4_heading_codes(quality_snapshot["heading_reference"])
    quality_raw_filter = hs4_mongodb_filter(quality_scope)
    quality_standard_filter = quality_raw_filter.copy()
    quality_standard_filter["cmdCode"] = {"$in": sorted(quality_heading_codes)}
    collection_quality = db[HS4_RAW_COLLECTION]
    quality_match = {"$match": quality_raw_filter}
    print("Read-only checks:", db.name, collection_quality.name)
else:
    print("Data quality checks skipped. No database access in this section.")

Read-only checks: swiss_trade_v2 hs4_raw


### Check observation counts and duplicate keys

First, check whether any stored documents fall outside the frozen reporting dimensions and selected partner areas.
Then count standard headings and nonstandard source codes by year.
The expected standard counts are 23,805, 24,188, and 24,376 for 2023, 2024, and 2025.
The expected nonstandard count is 30 rows of `9999` per year.

MongoDB already requires unique `_id` values within a collection.
We also group by the source fields defining one observation to detect duplicate observations with different identifiers.
A duplicate key has more than one stored document.
The duplicate count reports duplicate groups, not the number of excess documents.

In [16]:
if CHECK_HS4_DATA_QUALITY:
    stored_count = collection_quality.count_documents({})
    in_scope_count = collection_quality.count_documents(quality_raw_filter)
    print("Stored raw documents:", stored_count)
    print("Documents outside the selected source dimensions:", stored_count - in_scope_count)

    group_counts = {"$group": {
        "_id": {
            "period": "$period",
            "standardHeading": {"$in": ["$cmdCode", sorted(quality_heading_codes)]}
        },
        "observations": {"$sum": 1}
    }}
    sort_counts = {"$sort": {"_id.period": 1, "_id.standardHeading": -1}}
    pipeline = [quality_match, group_counts, sort_counts]
    quality_counts = list(collection_quality.aggregate(pipeline))
    print(pd.json_normalize(quality_counts).to_string(index=False))

    match_nonstandard = {"$match": {
        **quality_raw_filter, "cmdCode": {"$nin": sorted(quality_heading_codes)}
    }}
    group_codes = {"$group": {"_id": "$cmdCode", "observations": {"$sum": 1}}}
    pipeline = [match_nonstandard, group_codes, {"$sort": {"_id": 1}}]
    print("Nonstandard source codes:")
    pprint(list(collection_quality.aggregate(pipeline)))

    observation_fields = [
        "reporterCode", "period", "partnerCode", "partner2Code", "cmdCode",
        "classificationCode", "flowCode", "customsCode", "motCode"
    ]
    group_keys = {"$group": {
        "_id": {field: "$" + field for field in observation_fields},
        "documents": {"$sum": 1}
    }}
    match_duplicates = {"$match": {"documents": {"$gt": 1}}}
    pipeline = [quality_match, group_keys, match_duplicates, {"$count": "duplicateGroups"}]
    duplicate_result = list(collection_quality.aggregate(pipeline))
    print("Duplicate observation groups:", duplicate_result or [{"duplicateGroups": 0}])

Stored raw documents: 72459
Documents outside the selected source dimensions: 0
 observations _id.period  _id.standardHeading
        23805       2023                 True
           30       2023                False
        24188       2024                 True
           30       2024                False
        24376       2025                 True
           30       2025                False
Nonstandard source codes:
[{'_id': '9999', 'observations': 90}]
Duplicate observation groups: [{'duplicateGroups': 0}]


#### Count results and code `9999`

The counts match the expected dataset: 72,459 raw observations, including 72,369 standard HS4 observations.
MongoDB finds no out-of-scope documents or duplicate observation groups.

Code `9999` represents commodities not specified according to kind, not a standard H6 heading.
The source returns one such observation per selected partner and year: 90 rows in total.
These are actual source rows, not null values or missing observations.
They remain in `hs4_raw` but stay outside standard-heading analysis.

### Check field types, missing fields, and null values

Check every source field and `_id` across all raw documents, including rows outside the intended selection.
Each pipeline groups documents by one field's BSON type and counts them.
The Python loop only repeats the same MongoDB check for different field names.
It does not transform observations or calculate their totals.

Expect strings for HS codes and identifiers, numeric types for measurements, and booleans for most quality flags.
Keep `null` measurements unchanged; do not replace them with zero.
A field can legitimately contain more than one numeric BSON type.
Inspect unexpected types before deciding whether any conversion is necessary.

In [17]:
if CHECK_HS4_DATA_QUALITY:
    quality_fields = sorted({
        field for record in quality_snapshot["records"] for field in record
    } | {"_id"})
    quality_types = []
    for field in quality_fields:
        group_types = {"$group": {
            "_id": {"$type": "$" + field}, "observations": {"$sum": 1}
        }}
        sort_types = {"$sort": {"_id": 1}}
        pipeline = [group_types, sort_types]
        for result in collection_quality.aggregate(pipeline):
            quality_types.append({
                "field": field, "bsonType": result["_id"],
                "observations": result["observations"]
            })
    print(pd.DataFrame(quality_types).to_string(index=False))

                   field bsonType  observations
                     _id   string         72459
               aggrLevel      int         72459
                  altQty   double         72459
          altQtyUnitAbbr   string         72459
          altQtyUnitCode      int         72459
                cifvalue     null         72459
      classificationCode   string         72459
classificationSearchCode   string         72459
                 cmdCode   string         72459
                 cmdDesc   string         72459
             customsCode   string         72459
             customsDesc   string         72459
                flowCode   string         72459
                flowDesc   string         72459
                fobvalue   double         72459
                freqCode   string         72459
                grossWgt   double         72459
             isAggregate     bool         72459
       isAltQtyEstimated     bool         72459
     isGrossWgtEstimated     bool       

#### Null values and CIF valuation

`netWgt` contains 495 null values; the other 71,964 observations contain numeric net weights.
Null means unavailable, not zero weight.

All `cifvalue` entries are null: these export records provide no CIF valuation.

| Valuation | Meaning | Usual application |
|---|---|---|
| CIF: Cost, Insurance, and Freight | Includes goods, insurance, and freight to the importing country's border | Imports |
| FOB: Free On Board | Values goods at the exporting country's border, excluding international freight and insurance beyond it | Exports |

The range summary displays `NaN` for CIF because it contains no numeric observations.
These display values do not replace MongoDB's stored nulls.

### Check measurement ranges, units, and source flags

Check numeric ranges within the intended raw selection, including `9999`.
`$expr` lets `$match` use an aggregation expression to select numeric BSON types before checking their ranges.
`$cond` returns 1 when a measurement meets the condition and 0 otherwise.
`$sum` adds these indicators to count negative values and zero values.
The type table separately reports missing, null, and nonnumeric values.
Minimum and maximum values describe available numeric measurements, not missing measurements.

Trade values use current US dollars; net and gross weight use kilograms.
Quantity units vary by heading, so do not add quantities across incompatible units.
Zero measurements and estimation flags require interpretation; neither alone proves measured zero trade.
Keep reported, aggregate, and estimation flags as source information rather than automatic deletion rules.

Review these results before writing the field mapping and transformation.
This section does not create `trade_clean` or claim that the live checks have passed.

In [18]:
if CHECK_HS4_DATA_QUALITY:
    measurement_fields = [
        "primaryValue", "fobvalue", "cifvalue", "netWgt", "grossWgt", "qty", "altQty"
    ]
    quality_ranges = []
    for field in measurement_fields:
        match_numeric = {"$match": {
            "$expr": {"$in": [{"$type": "$" + field}, ["int", "long", "double", "decimal"]]}
        }}
        group_range = {"$group": {
            "_id": None, "numericObservations": {"$sum": 1},
            "minimum": {"$min": "$" + field}, "maximum": {"$max": "$" + field},
            "negativeValues": {"$sum": {"$cond": [{"$lt": ["$" + field, 0]}, 1, 0]}},
            "zeroValues": {"$sum": {"$cond": [{"$eq": ["$" + field, 0]}, 1, 0]}}
        }}
        pipeline = [quality_match, match_numeric, group_range]
        results = list(collection_quality.aggregate(pipeline))
        quality_ranges.append({"field": field, **(results[0] if results else {"numericObservations": 0})})
    print(pd.DataFrame(quality_ranges).drop(columns="_id", errors="ignore").to_string(index=False))

    group_units = {"$group": {
        "_id": {field: "$" + field for field in [
            "qtyUnitCode", "qtyUnitAbbr", "altQtyUnitCode", "altQtyUnitAbbr"
        ]},
        "observations": {"$sum": 1}
    }}
    pipeline = [quality_match, group_units, {"$sort": {"observations": -1, "_id": 1}}]
    print("Quantity unit combinations:")
    print(pd.json_normalize(list(collection_quality.aggregate(pipeline))).to_string(index=False))

    quality_flags = []
    for field in [
        "isOriginalClassification", "isLeaf", "isReported", "isAggregate",
        "isNetWgtEstimated", "isGrossWgtEstimated", "isQtyEstimated",
        "isAltQtyEstimated", "legacyEstimationFlag"
    ]:
        group_flags = {"$group": {
            "_id": {"type": {"$type": "$" + field}, "value": "$" + field},
            "observations": {"$sum": 1}
        }}
        pipeline = [quality_match, group_flags, {"$sort": {"_id": 1}}]
        for result in collection_quality.aggregate(pipeline):
            quality_flags.append({"field": field, **result["_id"], "observations": result["observations"]})
    print(pd.DataFrame(quality_flags).to_string(index=False))

       field  numericObservations  minimum      maximum  negativeValues  zeroValues
primaryValue                72459      1.1 5.367610e+10             0.0         0.0
    fobvalue                72459      1.1 5.367610e+10             0.0         0.0
    cifvalue                    0      NaN          NaN             NaN         NaN
      netWgt                71964      0.0 1.605252e+09             0.0        98.0
    grossWgt                72459      0.0 0.000000e+00             0.0     72459.0
         qty                72459      0.0 1.605252e+09             0.0      9531.0
      altQty                72459      0.0 3.470727e+09             0.0     43029.0
Quantity unit combinations:
 observations  _id.qtyUnitCode _id.qtyUnitAbbr  _id.altQtyUnitCode _id.altQtyUnitAbbr
        31000                8              kg                  -1                N/A
        21627                8              kg                   8                 kg
         9449               -1            

#### Measurement results

The checked numeric measurements contain no negative values.
Every raw observation has positive `primaryValue` and `fobvalue` values.
Their matching ranges do not prove equality for every observation.
The analysis will use `primaryValue` as its main measure.

| Finding | Treatment |
|---|---|
| All 72,459 `grossWgt` values equal zero | Do not use gross weight to measure export volume |
| 98 numeric `netWgt` values equal zero | Preserve them without assuming measured zero weight |
| Quantity units vary | Keep unit codes and labels; do not sum incompatible units |
| Some unit codes equal `-1`, with `N/A` | Do not use these quantities as meaningful physical amounts |

#### Source flags

Source flags describe how UN Comtrade classifies observations or estimates measurements.
All rows have `isAggregate = True`, `isLeaf = False`, and `isReported = False`.
These flags are consistent with HS4 summaries that combine finer product detail.
`isReported = False` does not mean no exports occurred or that every trade value is an estimate.

The source marks 40,610 net weights and 39,766 quantities as estimated.
The clean model will preserve these flags and retain `legacyEstimationFlag` without assigning undocumented meanings to its codes.

#### Transformation decisions

The checks do not justify removing additional observations or changing measurement values.
The planned clean collection will retain all 72,369 standard HS4 observations, including their nulls, zeros, units, and quality flags.
The analysis will focus on trade values rather than claims about physical-volume growth.

Next, review the proposed document model and draft UML diagram before defining the transformation.

## Agreed embedded data model

We agreed on these readable field names on 17 October 2026.
This section records the modeling decision; the data transformation and clean collection do not exist yet.
`exportingCountry` identifies the reporter, not the goods' production origin.
`destinationArea` includes territories and statistical partner areas, including code `490`, `Other Asia, nes`.
`tradeMeasures` distinguishes monetary values, weights, and quantities without calling monetary values trade volume.

### From extracted observations to clean documents

The Comtrade response supplies a list of flat HS4 observations.
The frozen snapshot stores these observations alongside query settings, the H6 reference, partner selection, and extraction logs.
The raw loader copies each observation into `hs4_raw` and adds a stable `_id`.

```mermaid
flowchart TD
    API["Comtrade response: list of flat HS4 observations"]
    API --> SNAP["Frozen JSON snapshot: records, settings, reference, and logs"]
    SNAP --> RAW["hs4_raw: one flat document per observation"]
    RAW --> MATCH["Select 72,369 standard H6 observations"]
    MATCH --> CLEAN["Planned trade_clean: one embedded document per observation"]
```

The planned transformation reads the raw MongoDB documents, not the entire snapshot container.
The saved H6 reference identifies standard headings and supplies classification labels.
`partner_totals_raw` remains historical partner-selection evidence; it does not supply clean product observations.

One clean document represents one year, reporter, partner area, and HS4 heading, with the other reporting dimensions fixed.
The main clean collection will retain all 72,369 standard observations, not only the top-50 subset.
Keep the 90 nonstandard `9999` observations in raw data only.
Keep the optional top-50 drill-down at its 4,235 observed combinations without inventing absent combinations.

### Why this model uses embedded objects

The course's Embedded Data Models section recommends keeping related information together within one document.
Currently, the raw document stores partner fields at its top level:

```json
{
    "partnerCode": 842,
    "partnerISO": "USA",
    "partnerDesc": "USA"
}
```

The agreed model groups these fields inside a `destinationArea` object within the same observation:

```json
{
    "destinationArea": {
        "comtradeCode": 842,
        "isoCode": "USA",
        "name": "USA"
    }
}
```

These examples show only the partner fields, not complete documents.
The same approach groups exporting-country, trade-flow, product, measurement, and quality fields into embedded objects.
MongoDB queries these fields through dot notation, such as `destinationArea.comtradeCode` and `tradeMeasures.primaryValueUSD`.

Embedding does not require arrays or one large document containing every observation for a partner.
Each observation remains a separate document, with its related information available without joining other collections.
This repeats some labels, but keeps ordinary analytical queries simple.
Embedding alone does not guarantee faster aggregation.

### Draft UML class diagram

This diagram records the agreed fields and types, not an implemented clean collection.

```mermaid
classDiagram
    class TradeObservation {
        _id : string
        year : string
    }
    class ExportingCountry {
        comtradeCode : int
        isoCode : string
        name : string
    }
    class DestinationArea {
        comtradeCode : int
        isoCode : string
        name : string
    }
    class TradeFlow {
        code : string
        description : string
    }
    class Product {
        classificationEdition : string
        aggregationLevel : int
        hsCode : string
        description : string
    }
    class Chapter {
        hsCode : string
    }
    class TradeMeasures {
        primaryValueUSD : double
        fobValueUSD : double
        netWeightKg : double or null
        quantity : double
        quantityUnitCode : int
        quantityUnit : string
        alternativeQuantity : double
        alternativeQuantityUnitCode : int
        alternativeQuantityUnit : string
    }
    class DataQuality {
        isNetWeightEstimated : bool
        isQuantityEstimated : bool
        isAlternativeQuantityEstimated : bool
        isGrossWeightEstimated : bool
        isOriginalClassification : bool
        isAggregate : bool
        isLeaf : bool
        isReported : bool
        legacyEstimationFlag : int
    }

    TradeObservation "1" *-- "1" ExportingCountry : exportingCountry
    TradeObservation "1" *-- "1" DestinationArea : destinationArea
    TradeObservation "1" *-- "1" TradeFlow : tradeFlow
    TradeObservation "1" *-- "1" Product : product
    TradeObservation "1" *-- "1" TradeMeasures : tradeMeasures
    TradeObservation "1" *-- "1" DataQuality : dataQuality
    Product "1" *-- "1" Chapter : chapter
```

The diamonds show composition: each embedded object forms part of its observation document.
Only `TradeObservation` represents documents in the planned `trade_clean` collection.
The other boxes represent embedded objects, not separate MongoDB collections or Python classes.
The `1` labels describe one object of each kind within each observation.
They do not mean that a partner appears in only one observation across the collection.

The course's UML examples provide the notation for these relationships.
`double or null` allows a numeric net weight or an explicit MongoDB null.
The final diagram must match the verified clean collection.

### How the model supports analysis

| Object or field | Purpose |
|---|---|
| `_id`, `year` | Trace observations to raw documents and compare years |
| `exportingCountry`, `destinationArea`, `tradeFlow` | Identify Switzerland, destination partner areas, and export direction |
| `product` | Identify HS4 headings and group them by parent HS2 chapter |
| `tradeMeasures` | Retain current-US-dollar values, net weight, quantities, and units |
| `dataQuality` | Retain source classification, reporting, aggregation, and estimation flags |

#### Agreed source-to-clean field mapping for data modeling

Each row copies the source value unchanged, except the derived chapter prefix.

| Raw field or expression | Clean field |
|---|---|
| `_id` | `_id` |
| `period` | `year` |
| `reporterCode` | `exportingCountry.comtradeCode` |
| `reporterISO` | `exportingCountry.isoCode` |
| `reporterDesc` | `exportingCountry.name` |
| `partnerCode` | `destinationArea.comtradeCode` |
| `partnerISO` | `destinationArea.isoCode` |
| `partnerDesc` | `destinationArea.name` |
| `flowCode` | `tradeFlow.code` |
| `flowDesc` | `tradeFlow.description` |
| `classificationCode` | `product.classificationEdition` |
| `aggrLevel` | `product.aggregationLevel` |
| `cmdCode` | `product.hsCode` |
| `cmdDesc` | `product.description` |
| First two characters of `cmdCode` | `product.chapter.hsCode` |
| `primaryValue` | `tradeMeasures.primaryValueUSD` |
| `fobvalue` | `tradeMeasures.fobValueUSD` |
| `netWgt` | `tradeMeasures.netWeightKg` |
| `qty` | `tradeMeasures.quantity` |
| `qtyUnitCode` | `tradeMeasures.quantityUnitCode` |
| `qtyUnitAbbr` | `tradeMeasures.quantityUnit` |
| `altQty` | `tradeMeasures.alternativeQuantity` |
| `altQtyUnitCode` | `tradeMeasures.alternativeQuantityUnitCode` |
| `altQtyUnitAbbr` | `tradeMeasures.alternativeQuantityUnit` |
| `isNetWgtEstimated` | `dataQuality.isNetWeightEstimated` |
| `isQtyEstimated` | `dataQuality.isQuantityEstimated` |
| `isAltQtyEstimated` | `dataQuality.isAlternativeQuantityEstimated` |
| `isGrossWgtEstimated` | `dataQuality.isGrossWeightEstimated` |
| `isOriginalClassification` | `dataQuality.isOriginalClassification` |
| `isAggregate` | `dataQuality.isAggregate` |
| `isLeaf` | `dataQuality.isLeaf` |
| `isReported` | `dataQuality.isReported` |
| `legacyEstimationFlag` | `dataQuality.legacyEstimationFlag` |

Keep `_id`, `year`, and HS codes as strings, including leading zeros.
Keep country and unit codes as integers, measurements as doubles, and source booleans unchanged.
Keep `legacyEstimationFlag` as an integer.
Preserve nulls, zeros, unit labels, and official partner-area names without substitution.
Keep `netWeightKg` nullable and retain its estimate flag.

#### Fields outside the clean model

Omit all-null `cifvalue` and all-zero `grossWgt` from clean documents; preserve both fields in raw data.
Retain `isGrossWeightEstimated` to preserve the agreed source flags, despite omitting the gross-weight measurement.
Store only `product.chapter.hsCode`, not `chapter.description`.
Use the saved H6 reference to label chapter results after MongoDB calculates totals.
Keep fixed reporting dimensions in the snapshot and notebook settings rather than repeating them in clean documents.
The transformation filter still checks those dimensions.
The raw `_id` links each clean document to all original fields.

### Planned data transformation

The course exercises guide the implementation:

| Course section | Application here |
|---|---|
| Lecture 5: Data Model Design, page 200 | Embed related fields within each observation |
| Lecture 5: Updating Collection Structure, page 209 | Add nested fields and later save with `$out` |
| Lecture 6: arrayToObject, page 249 | Follow the named `project1` dictionary and `aggregate()` pattern |
| Lecture 6: Group And Push, page 251 | Use named stages and a readable Python pipeline list |
| Lecture 2: Class Diagram, page 59 | Show embedded objects through UML composition |

We reuse the code style, not every operator from each exercise.
These observations need no arrays, `$push`, `$arrayToObject`, separate entity collections, or joins.
Existing BSON types need no conversion, so we do not copy the string-to-integer conversion exercise.
We do not copy exercise resets such as `drop()` or `delete_many({})`.

```mermaid
flowchart TD
    RAW["hs4_raw: existing observations"] --> MATCH["match1: standard headings and frozen source dimensions"]
    MATCH --> PROJECT["project1: rename fields and embed objects"]
    PROJECT --> ADD["add_fields1: derive parent HS2 code"]
    ADD --> PREVIEW["Read-only preview and data quality checks"]
    PREVIEW --> APPROVAL["Review results and approve saving"]
    APPROVAL --> SAVE["Check destination, then save the clean collection"]
```

1. Start from the existing data quality setup and its frozen scope and standard-heading filter.
   Run only required imports, definitions, and connection cells when session variables are unavailable.
   Do not rerun historical extraction or loading cells.
2. Define `match1` with `$match` using `quality_standard_filter`.
   This retains all 72,369 standard HS4 observations within the frozen source dimensions and selected partner areas.
   Do not filter to the top 50 or delete observations based on reporting or aggregation flags.
3. Define `project1` with `$project` using the agreed mapping.
   Include `_id` explicitly and construct embedded objects with ordinary Python dictionaries.
   Use field expressions such as `"year": "$period"` to rename without converting values.
4. Define `add_fields1` with `$addFields` after the projection.
   Set `product.chapter.hsCode` to `{"$substrBytes": ["$product.hsCode", 0, 2]}`.
   `$substrBytes` extracts two bytes starting at position zero; standard HS codes contain only ASCII digits.
   This additional expression preserves the chapter's leading zero without integer conversion.
5. Build `pipeline = [match1, project1, add_fields1]`.
   Preview three documents with `collection.aggregate(pipeline + [{"$limit": 3}])` and display them with `pprint()`.
   Here, `collection` refers to the existing `hs4_raw` collection.
   This preview does not save documents or establish that the entire transformation is correct.
6. Run separate read-only data quality checks across the full transformed result.
   Check total and yearly counts, unique identifiers, embedded BSON types, nulls, zeros, units, flags, and leading zeros.
   Compare each retained field with its raw counterpart, not only matching ranges or sums.
   Use a validation-only pipeline that temporarily carries raw values beside transformed fields.
   Compare these pairs in MongoDB after transformation, then verify totals within the same standard-heading scope.
   Check primary and FOB values independently; do not assume they equal each other.
   Expect 23,805 observations for 2023, 24,188 for 2024, and 24,376 for 2025.
7. Review the preview and checks before approving any save.
   Check `db.list_collection_names()` before choosing `trade_clean` as the destination.
   `$out` can replace an existing collection; never replace one without explicit approval.
   After approval, add a separate named `$out` stage to the full pipeline, never the three-document preview.
8. Verify the saved collection, its nested fields, and the unchanged raw collection.
   Finalize the UML against the stored structure only after these checks pass.

For later HS2 analysis, group standard HS4 values by `year`, destination, and `product.chapter.hsCode` inside MongoDB.
Never add parent totals to their child-heading values.
Use Pandas and Matplotlib mainly to present MongoDB results.
Shares describe the selected standard-HS4 dataset, not all Swiss exports.
Current-US-dollar changes describe nominal export-value changes, not physical-volume growth.

This section documents the plan only; it adds no executable transformation or database write.

# Sources and method references

- [UN Comtrade API documentation](https://comtradeplus.un.org/ApiDocs): endpoints, query parameters, and authentication.
- [UN Comtrade API guide](https://uncomtrade.org/docs/un-comtrade-api/): public preview versus authenticated access.
- [Partner selection and World behavior](https://uncomtrade.org/docs/data/): the `ALL` selection and the World aggregate.
- [Subscription plans](https://uncomtrade.org/docs/subscriptions/): current free-tier record, call, and rate limits.
- [Partner-area reference](https://comtradeapi.un.org/files/v1/app/reference/partnerAreas.json): partner codes and descriptions.
- [H6 classification reference](https://comtradeapi.un.org/files/v1/app/reference/H6.json): source codes for HS 2022 heading membership.
- [WCO HS 2022 nomenclature](https://www.wcoomd.org/en/media/newsroom/2021/november/the-2022-edition-of-the-harmonized-system-nomenclature-is-now-available-online.aspx): HS hierarchy and code counts.
- [Local course notes](txt%20course%20content/Copy_paste_course_content.md): data modeling, PyMongo, and aggregation concepts.

Trade values are current US dollars and are not inflation-adjusted. Partner areas may include territories or aggregates. Source data may be revised.

# PDF export and final submission checklist

The first Markdown cell contains Quarto YAML settings. Quarto supplies the title, table of contents, and section numbering. Rendering is configured not to execute Python cells; the PDF uses saved outputs.

Render V2 from this notebook's folder with:

```bash
quarto render "UN_Comtrade_V2.ipynb" --to pdf --no-execute
```

Before final submission:

- Complete the MongoDB connection, raw/clean transformations, analysis, conclusions, and learnings.
- Include at least two images, including a class diagram that matches the implemented clean model.
- Restart the kernel, run all intended cells successfully, and save outputs. Inspect API rerun and database write behavior first.
- Check for stale outputs, credentials, errors, clipped tables/code, and missing images.
- Render and inspect every PDF page, then package the PDF, executed V2 notebook, and required images in the ILIAS ZIP.

Do not treat this discovery-stage notebook or its partner-summary CSV as the completed analysis.